# Saving & loading models

Training is expensive; inference should be cheap. Once you've trained, tuned,
and [explained](../07-explainability/model-interpretability.ipynb) a model, you
want to **persist** it — write it to disk and reload it later to serve
predictions without retraining.

In Rust the idiomatic path is [`serde`](https://serde.rs) for the
serialize/deserialize traits plus a binary format like
[`bincode`](https://docs.rs/bincode). Any model type that derives
`Serialize`/`Deserialize` can be saved.

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep bincode = { version = "1.3" }
:dep ndarray = { version = "0.16" }
:dep onnx-export-rs = { version = "0.1.1" }

// A trained model = its learned parameters. This mirrors the linear-regression
// chapter (weights + bias). Deriving serde traits makes it persistable.
#[derive(serde::Serialize, serde::Deserialize, Clone, Debug)]
struct LinearModel {
    weights: Vec<f64>,
    bias: f64,
}
impl LinearModel {
    fn predict(&self, x: &[f64]) -> f64 {
        self.bias + self.weights.iter().zip(x).map(|(w, xi)| w * xi).sum::<f64>()
    }
}

let model = LinearModel { weights: vec![2.0, -1.0], bias: 0.5 };
let sample = [3.0, 4.0];
println!("prediction before saving = {:.3}", model.predict(&sample));

## Round-trip through disk

Serialize to bytes, write to a file, read it back, deserialize — then confirm
the reloaded model makes the **identical** prediction:

In [ ]:
{
    let path = "/tmp/linear_model.bin";
    // Save
    let bytes = bincode::serialize(&model).unwrap();
    std::fs::write(path, &bytes).unwrap();
    println!("wrote {} bytes to {}", bytes.len(), path);
    // Load
    let loaded_bytes = std::fs::read(path).unwrap();
    let loaded: LinearModel = bincode::deserialize(&loaded_bytes).unwrap();
    println!("prediction after loading = {:.3}", loaded.predict(&sample));
    println!("match: {}", (model.predict(&sample) - loaded.predict(&sample)).abs() < 1e-12);
}

## Versioning models on disk

Don't silently overwrite a working model. Save **metadata** alongside it — a
version tag, the training date, and something about the data (row count or a
hash) — so you can tell models apart and roll back:

In [ ]:
#[derive(serde::Serialize, serde::Deserialize, Debug)]
struct ModelArtifact {
    version: u32,
    trained_on: String,
    n_train_rows: usize,
    model: LinearModel,
}

{
    let artifact = ModelArtifact {
        version: 1,
        trained_on: "2026-01-15".to_string(),
        n_train_rows: 5000,
        model: model.clone(),
    };
    let bytes = bincode::serialize(&artifact).unwrap();
    std::fs::write("/tmp/model_v1.bin", &bytes).unwrap();
    let reloaded: ModelArtifact = bincode::deserialize(&std::fs::read("/tmp/model_v1.bin").unwrap()).unwrap();
    println!("loaded artifact v{} trained {} on {} rows",
             reloaded.version, reloaded.trained_on, reloaded.n_train_rows);
}

## Interoperability: ONNX

`serde` + `bincode` keeps a model inside the Rust world. **ONNX** is the
cross-ecosystem exchange format, and it works **both** directions:

- **Import** — [`tract`](https://docs.rs/tract-onnx) loads and runs `.onnx` files
  directly in Rust, so a model trained in PyTorch or scikit-learn (via `skl2onnx`)
  can be served from a Rust binary.
- **Export** — [`onnx-export-rs`](https://crates.io/crates/onnx-export-rs) writes
  canonical Rust models (linear / logistic / tree / forest / SVM / k-means / …) to
  `.onnx`, so a model *trained in Rust* can run in any ONNX runtime elsewhere —
  Python, C++, the browser, or an edge device.

A linear model is just its weights and bias, which is exactly what a canonical
ONNX linear graph needs. We export the same `model` from above:

In [ ]:
use ndarray::Array1;
use onnx_export_rs::{canonical::LinearModelWeights, exporters::export_linear, save_to_file};

{
    // Canonical weights come straight from the trained model's parameters.
    let weights = LinearModelWeights::new(Array1::from(model.weights.clone()), model.bias);
    let path = "/tmp/linear_model.onnx";
    save_to_file(&export_linear(&weights), path).unwrap();
    let sz = std::fs::metadata(path).unwrap().len();
    println!("exported the model to {path}  ({sz} bytes of ONNX)");
    println!("it emits a `Gemm` op (opset 13) and can now be loaded by tract, onnxruntime, etc.");
}

```{note}
Which persistence format to reach for: **`serde` + `bincode`** is the reliable
default when both ends are Rust — it's exact, fast, and preserves the whole model
struct. **ONNX** (`onnx-export-rs` out, `tract` in) is the choice when you need to
**cross ecosystems** — hand a Rust-trained model to a Python/JS/edge runtime, or
run a Python-trained model inside Rust. Note that ONNX narrows parameters to
`f32`, so for a Rust-only pipeline `bincode` keeps full `f64` precision. Some
`smartcore`/`linfa` internals aren't public, so `onnx-export-rs` also offers
opt-in adapters that read those crates' serialized state.
```

Next: [serving a model](serving-a-model.ipynb) — loading a persisted model behind
an HTTP endpoint.